# 9. Spelling keys and search

Can rule-based spelling keys beat a lookup table, and does spelling normalization help someone search the transcripts?

In [1]:
from nbutil import *
import pandas as pd

## Spelling keys on the labelled pairs
Precision and recall (%) for every detector, on all pairs and on pairs the lookup table has seen or never seen.

In [2]:
K = pd.read_csv(ROOT / "results/spelling_keys/summary.csv", encoding="utf-8-sig")
sub = K[K["subset"].isin(["all", "seen", "unseen"])][["subset", "detector", "spelling", "precision", "recall"]]
sub.assign(precision=(100 * sub["precision"]).round(1), recall=(100 * sub["recall"]).round(1))

,subset,detector,spelling,precision,recall
0,all,table,381,90.3,58.8
1,all,fold,381,98.2,57.0
2,all,deep,381,97.8,69.3
3,all,uzzaman04,381,96.2,80.3
28,seen,table,345,90.3,64.9
29,seen,fold,345,98.1,59.1
30,seen,deep,345,97.6,70.1
31,seen,uzzaman04,345,96.5,80.6
32,unseen,table,36,NaN,0.0
33,unseen,fold,36,100.0,36.1


## Keyword search
Mean average precision, system by test set. Exact match, then the fold (spelling-normalized) column.

In [3]:
R = pd.read_csv(ROOT / "results/retrieval/summary.csv", encoding="utf-8-sig")
kw = R[R["task"] == "keyword"]
for cond in ("exact", "fold"):
    t = kw[kw["condition"] == cond].pivot(index="system", columns="set", values="map")
    display(Markdown("**" + cond + "**"))
    display(t.reindex(columns=list(SETS)).rename(columns=SETS).round(1))

**exact**

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,71.2,80.5,90.5,91.8
IndicConformer CTC,65.3,76.4,79.7,88.2
M1,80.8,73.2,82.9,85.8
final,80.9,76.7,85.7,89.9
list correction,73.8,72.4,NaN,83.6
zero-shot,71.5,72.2,80.7,85.1


**fold**

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,72.8,81.4,90.4,91.4
IndicConformer CTC,67.3,77.4,80.5,88.3
M1,81.7,74.2,83.8,86.5
final,81.4,77.8,86.3,90.2
list correction,74.9,73.9,NaN,84.8
zero-shot,73.5,74.1,82.1,86.3


In [4]:
pd.read_csv(ROOT / "results/retrieval/predictions.csv", encoding="utf-8-sig").round(2)

,prediction,contrast,sets,diff,lo,hi
0,P26a,"final - zero-shot, exact",splits_fa-eval,9.36,7.19,11.47
1,P26a,"final - zero-shot, exact",fleurs-test,4.53,2.37,6.77
2,P26a,"final - zero-shot, exact",cv-test,5.03,3.24,6.79
3,P26a,"final - zero-shot, exact",kathbath-valid,4.80,3.06,6.58
4,P26b,fold gain: zero-shot - final,splits_fa-eval,1.38,0.61,2.30
5,P26b,fold gain: zero-shot - final,splits_fa-eval+fleurs-test+cv-test+kathbath-valid,0.98,0.51,1.48
6,P26c,final: fold - exact,splits_fa-eval,0.56,0.14,1.10
7,P26c,final: fold - exact,fleurs-test,1.14,0.38,2.05
8,P26c,final: fold - exact,cv-test,0.60,0.21,1.07
9,P26c,final: fold - exact,kathbath-valid,0.24,-0.39,0.88


## Is spelling normalization equivalent to no gain?
TOST on the per-term MAP difference (fold minus exact), margin ±1 point.

In [5]:
T = pd.read_csv(ROOT / "results/robustness/tost.csv", encoding="utf-8-sig")
for c in ("mean_d", "ci90_lo", "ci90_hi", "ci95_lo", "ci95_hi"):
    T[c] = T[c].round(2)
T

,set,system,n_terms,mean_d,ci90_lo,ci90_hi,ci95_lo,ci95_hi,margin,equivalent,prediction
0,splits_fa-eval,final,500,0.56,0.18,1.01,0.14,1.10,1.0,False,P29a
1,splits_fa-eval,zero-shot,500,1.95,1.16,2.83,1.04,2.99,1.0,False,P29b
2,fleurs-test,final,500,1.14,0.47,1.84,0.38,2.00,1.0,False,NaN
3,fleurs-test,zero-shot,500,1.91,1.01,2.86,0.86,3.06,1.0,False,NaN
4,cv-test,final,500,0.60,0.27,0.98,0.23,1.07,1.0,True,P29a
5,cv-test,zero-shot,500,1.44,0.68,2.30,0.58,2.50,1.0,False,NaN
6,kathbath-valid,final,500,0.24,-0.28,0.74,-0.38,0.84,1.0,True,P29a
7,kathbath-valid,zero-shot,500,1.18,0.30,2.12,0.14,2.32,1.0,False,NaN


## Word families
Mean average precision when the query matches any inflected form, and the paired differences.

In [6]:
F = pd.read_csv(ROOT / "results/retrieval_family/summary.csv", encoding="utf-8-sig")
fam = F[F["condition"] == "family"].pivot(index="system", columns="set", values="map")
display(fam.reindex(columns=list(SETS)).rename(columns=SETS).round(1))
pd.read_csv(ROOT / "results/retrieval_family/predictions.csv", encoding="utf-8-sig").round(2)

set,DL Sprint eval,FLEURS,Common Voice,Kathbath
system,,,,
Hishab,74.9,82.6,92.1,93.5
IndicConformer CTC,69.7,78.8,81.6,90.4
M1,83.6,75.4,85.2,88.7
final,83.2,79.0,87.5,92.1
list correction,76.6,75.3,NaN,87.1
zero-shot,75.1,75.5,83.3,88.4


,prediction,contrast,sets,diff,lo,hi
0,P28a,"final - zero-shot, family",splits_fa-eval,8.16,6.33,9.87
1,P28a,"final - zero-shot, family",fleurs-test,3.48,1.58,5.39
2,P28a,"final - zero-shot, family",cv-test,4.26,2.78,5.83
3,P28a,"final - zero-shot, family",kathbath-valid,3.73,2.38,5.19
4,P28b,family gain: zero-shot - final,splits_fa-eval,-1.40,-2.46,-0.24
5,P28b,family gain: zero-shot - final,splits_fa-eval+fleurs-test+cv-test+kathbath-valid,-0.16,-0.70,0.41
6,P28c,final: family - exact,splits_fa-eval,25.56,22.72,28.31
7,P28c,final: family - exact,fleurs-test,13.92,11.55,16.36
8,P28c,final: family - exact,cv-test,25.49,22.73,28.32
9,P28c,final: family - exact,kathbath-valid,25.12,22.24,27.62


**Conclusion:** rule keys find spellings of unseen words (the lookup table finds 0 of 36). The 2004 key is best.
Fine-tuning makes transcripts more searchable (+4.5 to +9.4 MAP). After fine-tuning, spelling normalization is
equivalent to no gain on Common Voice and Kathbath (TOST, ±1 MAP). Word families add 14-26 points for every system:
a search problem, not the recognizer's.